[◀ Notebook 18: Testing & Quality](18_testing_and_code_quality.ipynb) | [Table of Contents](TOC.ipynb) | [Appendix A: DB Connectivity ▶](Appendix_A_database_connectivity.ipynb)

# Notebook 19: Memory Management, Weak References & Garbage Collection

Welcome to Notebook 19 of the Bottom-Up Python Tutorial. In this notebook, we explore the under-the-hood execution mechanics of CPython's memory management. We will cover reference counting, the cyclic garbage collector (generations and thresholds), how to use weak references (`weakref`) to avoid reference cycles in caches, and how to profile memory and execution using `tracemalloc` and `cProfile`.

### References:
- [Python standard library: gc (Garbage Collector)](https://docs.python.org/3/library/gc.html)
- [Python standard library: weakref](https://docs.python.org/3/library/weakref.html)
- [Python standard library: tracemalloc](https://docs.python.org/3/library/tracemalloc.html)
- [Python standard library: cProfile](https://docs.python.org/3/library/profile.html)

## 1. Reference Counting in CPython

In CPython, every object is represented by a C struct (`PyObject`) that contains a reference count field (`ob_refcnt`).

- **Incrementing:** When you bind a name to an object, add it to a list, or pass it to a function, CPython increments its reference count.
- **Decrementing:** When a variable goes out of scope, is reassigned, or deleted, CPython decrements the reference count.
- **Deallocation:** The moment an object's reference count reaches **0**, its memory is reclaimed immediately.

### Inspecting Refcounts
You can inspect an object's reference count using `sys.getrefcount(obj)`. Note that passing the object to `sys.getrefcount` creates a temporary reference, so the returned count is always **1 higher** than the actual count.

In [ ]:
import sys

a = [1, 2, 3]
print("Initial refcount:", sys.getrefcount(a) - 1)  # Subtract 1 for the temp argument ref

b = a
print("Refcount after assigning b = a:", sys.getrefcount(a) - 1)

c = [a, a]
print("Refcount after adding to list twice:", sys.getrefcount(a) - 1)

del b
print("Refcount after deleting b:", sys.getrefcount(a) - 1)

## 2. The Cyclic Garbage Collector

Reference counting cannot detect **reference cycles** (circular references). For example:
```python
x = {}
y = {}
x['friend'] = y
y['friend'] = x
del x
del y
```
Even though we deleted `x` and `y` from our global namespace, they still reference each other. Their reference counts remain at **1**, meaning they will leak memory and never be freed by reference counting alone.

### The Generational GC
To solve this, CPython runs a cyclic garbage collector in the background (using the `gc` module):
- **Generations:** Objects are split into 3 generations (0, 1, 2). New objects start in Gen 0. If they survive a GC collection, they are promoted to Gen 1, and eventually Gen 2.
- **Thresholds:** Each generation has a threshold. When the number of allocations minus deallocations in Gen 0 exceeds the threshold, Gen 0 collection is triggered. If Gen 1/2 triggers are met, they are also collected.
- **Cost:** Cyclic GC sweeps are slow and cause "stop-the-world" pauses, so CPython only runs them periodically or when thresholds are exceeded.

In [ ]:
import gc

# Check GC thresholds for Gen 0, 1, 2
print("GC Thresholds:", gc.get_threshold())

# Check if GC is enabled
print("Is GC enabled?", gc.isenabled())

# Force manual collection of all generations
unreachable_count = gc.collect()
print(f"Manually collected {unreachable_count} unreachable objects.")

## 3. Weak References (`weakref`)

A **weak reference** allows you to reference an object *without* incrementing its reference count. This is extremely useful for implementing caches, observers, or registry maps, where holding onto an object in the cache should not prevent it from being garbage collected if it is no longer referenced anywhere else.

- `weakref.ref(obj)`: Creates a weak reference object. Calling it like a function `ref()` returns the referenced object if it is still alive; otherwise, it returns `None`.
- `weakref.WeakValueDictionary`: A dictionary implementation where values are held weakly. If a value object has no other strong references left, it is automatically removed from this dictionary.

### Caching with `WeakValueDictionary` and Garbage Collection Mechanics

#### 1. The Strong Reference Problem in Standard Caches
When you use a standard Python `dict` as a cache, the dictionary holds a **strong reference** to every value. Even if the rest of your application discards all references to an object, the dictionary's reference keeps the object's refcount at least at `1`. Consequently, the object remains allocated in memory forever (or until the dictionary itself is deleted). This is a classical memory leak scenario.

#### 2. How `WeakValueDictionary` Automates Pruning
`weakref.WeakValueDictionary` solves this by storing values as weak references under the hood:
- **Callback Registration**: When a value is inserted into a `WeakValueDictionary`, the dictionary wraps the value in a `weakref.ref` object and registers a **destruction callback** with it.
- **Deallocation Flow**: When all external strong references to the value object are gone, its reference count drops to `0` (or the cyclic GC detects it as unreachable).
- **Automatic Pruning**: During the object's deallocation phase, CPython executes the weak reference callback. This callback automatically deletes the corresponding key and the weak reference from the `WeakValueDictionary` internal hash table. 
- **Result**: The dictionary does not grow unbounded; it dynamically cleans itself up in real-time as objects are naturally deallocated.

#### 3. Types that Support Weak References
Not all Python objects can be weakly referenced. To optimize memory layout, CPython does not allocate weak reference slots for several built-in types:
- **Unsupported types**: Raw `list`, `dict`, `set`, `tuple`, `int`, `float`, and `complex` do not support weak references.
- **Supported types**: User-defined class instances (unless they define `__slots__` without including `__weakref__`), subclasses of built-in types, and standard library instances (like `set` subclasses or custom objects) support weak references.

### Concrete Code Example: WeakValueDictionary Cache Behavior

The code snippet below demonstrates how a `WeakValueDictionary` automatically prunes key-value pairs the moment external references are dropped:

```python
import weakref
import gc

class HeavyData:
    def __init__(self, value):
        self.value = value

# Create the weak value cache
cache = weakref.WeakValueDictionary()

# 1. Create a strong reference to an object
external_ref = HeavyData(42)
cache["item1"] = external_ref

# 2. Put another object in without saving a long-term strong reference
cache["item2"] = HeavyData(99)

print("Before GC:")
print("  item1 in cache:", "item1" in cache)  # True
print("  item2 in cache:", "item2" in cache)  # True

# 3. Force garbage collection. item2 has no external strong references,
# while item1 is still pointed to by external_ref.
gc.collect()

print("After GC:")
print("  item1 in cache:", "item1" in cache)  # True
print("  item2 in cache:", "item2" in cache)  # False (Automatically pruned!)
```

In [ ]:
import weakref
import gc

class HeavyObject:
    def __init__(self, name):
        self.name = name

obj = HeavyObject("BigData")
wref = weakref.ref(obj)

print("Weak reference target:", wref())

# Delete the only strong reference to obj
del obj

# Run GC to ensure deallocation
gc.collect()

print("Weak reference target after deletion:", wref())

## 4. Profiling Memory & Performance

- **`tracemalloc`:** A built-in module to track memory allocations. It can take snapshots of memory, letting you compare them before and after running code to find memory leaks.
- **`cProfile`:** A built-in execution profiler that counts how many times functions are called and how long they take.

In [ ]:
import tracemalloc

tracemalloc.start()
snapshot1 = tracemalloc.take_snapshot()

# Allocate some memory
temp_list = [i for i in range(100_000)]

snapshot2 = tracemalloc.take_snapshot()
top_stats = snapshot2.compare_to(snapshot1, 'lineno')

print("Top memory allocations:")
for stat in top_stats[:3]:
    print(stat)

tracemalloc.stop()

---

## 5. Coding Challenges

Complete the two challenges below.

### Challenge 1: Memory-Safe Cache using WeakValueDictionary

Implement a cache class called `MemorySafeCache` that stores objects. 
It should use `weakref.WeakValueDictionary` internally to hold onto the objects. 
Your cache must expose two methods:
- `set(key, obj)`: Adds an object to the cache.
- `get(key)`: Returns the object if it exists in the cache, or `None` if it has been garbage collected or doesn't exist.

In [ ]:
import weakref

class MemorySafeCache:
    # TODO: Implement the memory safe cache using weakref.WeakValueDictionary
    def __init__(self):
        pass

    def set(self, key, obj):
        pass

    def get(self, key):
        pass

In [ ]:
# Solution Code for Challenge 1
# class MemorySafeCache:
#     def __init__(self):
#         self._cache = weakref.WeakValueDictionary()
# 
#     def set(self, key, obj):
#         self._cache[key] = obj
# 
#     def get(self, key):
#         return self._cache.get(key, None)

In [ ]:
# Challenge 1 Verification Test
import gc

class CacheableItem:
    def __init__(self, data):
        self.data = data

cache = MemorySafeCache()

# 1. Create a strong reference to an item
item1 = CacheableItem("Item 1")
cache.set("k1", item1)

# Verify that we can retrieve it
assert cache.get("k1") is item1, "Cache should return the item"

# 2. Put another item in without holding a long-term strong reference
cache.set("k2", CacheableItem("Item 2"))

# Collect garbage
gc.collect()

# Verify that k2 has been garbage-collected from the cache because there were no external strong references to it
assert cache.get("k2") is None, "Cache item k2 should have been garbage collected"
assert cache.get("k1") is item1, "Cache item k1 should still be present"

print("Challenge 1: Success! Memory-safe cache working as expected.")

### Challenge 2: Cyclic Reference and Memory Profiler with `tracemalloc`

Implement a circular reference using a `Node` class that has `parent` and `children` links. 
Write a function `create_leak()` that creates `10,000` nodes and structures them into reference cycles, then deletes local references to them. 
Write code using `tracemalloc` to verify that:
1. Calling `create_leak()` with `gc` disabled causes a memory leak (non-zero allocation size difference).
2. Calling `gc.collect()` clears the leaked memory.

In [ ]:
import gc
import tracemalloc

class Node:
    def __init__(self):
        self.parent = None
        self.children = []

def create_leak():
    # TODO: Create 10,000 node pairs where child.parent = parent and parent.children.append(child).
    # Delete all local references before returning.
    pass

In [ ]:
# Solution Code for Challenge 2
# def create_leak():
#     nodes = []
#     for _ in range(10000):
#         parent = Node()
#         child = Node()
#         child.parent = parent
#         parent.children.append(child)
#         nodes.append((parent, child))
#     # delete reference list
#     del nodes

In [ ]:
# Challenge 2 Verification Test
gc.enable()
gc.collect()  # Clean up before test

tracemalloc.start()
snapshot_start = tracemalloc.take_snapshot()

# Disable GC to prevent automatic cleanup of cycles during the test
gc.disable()

create_leak()

snapshot_after_leak = tracemalloc.take_snapshot()
diff_leak = snapshot_after_leak.compare_to(snapshot_start, 'lineno')
total_leak = sum(stat.size_diff for stat in diff_leak)
print(f"Memory leak size: {total_leak} bytes")

# Re-enable and force collection
gc.enable()
collected = gc.collect()
print(f"GC manually collected {collected} unreachable objects.")

snapshot_after_gc = tracemalloc.take_snapshot()
diff_after_gc = snapshot_after_gc.compare_to(snapshot_start, 'lineno')
total_after_gc = sum(stat.size_diff for stat in diff_after_gc)
print(f"Remaining memory diff: {total_after_gc} bytes")

tracemalloc.stop()

# Assertions
assert total_leak > 100_000, f"Expected significant leak (> 100KB), got {total_leak} bytes"
assert collected >= 20_000, f"Expected to collect at least 20,000 nodes, collected {collected}"
assert total_after_gc < total_leak / 2, f"Memory diff after GC should be substantially reduced, got {total_after_gc} bytes"
print("Challenge 2: Success! Leaked cyclic reference detected and reclaimed by GC.")

[◀ Notebook 18: Testing & Quality](18_testing_and_code_quality.ipynb) | [Table of Contents](TOC.ipynb) | [Appendix A: DB Connectivity ▶](Appendix_A_database_connectivity.ipynb)